In [ ]:
import pandas as pd
from src.download_data import get_data

In [ ]:
data = get_data()
data.head(5)

In [ ]:
# Validate nulls
print(data.info())
null_before_clean = data.isnull().sum()
display('-Validation nulls count:-', null_before_clean)

# Handle nulls
data.dropna(subset=['genres', 'categories'], inplace=True)
data.fillna('Unknown', inplace=True)
data.reset_index(drop=True, inplace=True)

null_after_clean = data.isnull().sum()
display('-Handled nulls count:-', null_after_clean)

In [ ]:
# Validate 'name' column
data['name'] = data['name'].str.strip()

In [ ]:
# Validate 'release_year' and  'release_date' columns
data_years = data['release_year'].unique()
study_years = [2021, 2022, 2023, 2024, 2025]
if sorted(data_years) != study_years:
    data = data[data['release_year'].isin(study_years)]
print(f"Unique years in 'release_year': {data_years}")

data['release_date'] = pd.to_datetime(data['release_date'], errors='coerce')
print(f"Number of missing release dates: {data['release_date'].isna().sum()}")
data.dropna(subset=['release_date'], inplace=True)

In [ ]:
# Validate 'genres' and 'categories' columns
genres_dummies = data['genres'].str.get_dummies(sep=';').astype('Sparse[uint8]')
categories_dummies = data['categories'].str.get_dummies(sep=';').astype('Sparse[uint8]')
data = pd.concat([data, genres_dummies, categories_dummies], axis=1)
data.head(5)

In [ ]:
# Validate 'price' and 'recommendations' columns
negative_prices = data['price'] < 0
negative_recommendations = data['recommendations'] < 0
data = data[~(negative_prices | negative_recommendations)]
print(f"Removed {negative_prices.sum()} negative prices and {negative_recommendations.sum()} negative recommendations")

In [ ]:
# Validate 'developer' and 'publisher' columns
data['developer'] = data['developer'].str.strip()
data['publisher'] = data['publisher'].str.strip()

In [ ]:
# Save validated data
print(f"Data final shape: {data.shape}")
data.to_csv('../data/validated_steam_games.csv', index=False)